In [14]:
import pandas as pd
import xlsxwriter
from dateutil.relativedelta import relativedelta
from datetime import datetime

In [15]:
ar_xl = pd.ExcelFile("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/ar.xlsx")
ar_sheets = {}
for sheet_name in ar_xl.sheet_names:
    ar_sheets[sheet_name] = ar_xl.parse(sheet_name) 

var_xl = pd.ExcelFile("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/var.xlsx")
var_sheets = {}
for sheet_name in var_xl.sheet_names:
    var_sheets[sheet_name] = var_xl.parse(sheet_name)

ml_xl = pd.ExcelFile("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/07_model_output/ml.xlsx")
ml_sheets = {}
for sheet_name in ml_xl.sheet_names:
    ml_sheets[sheet_name] = ml_xl.parse(sheet_name) 

In [16]:
variable = pd.read_csv("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/02_intermediate/variable.csv")
vintagedata = pd.read_csv("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/0_source/00_extract_vintagedata.csv")
ts = pd.read_parquet("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/02_intermediate/non_transformed_data.parquet")

vintagedata["ReferenceDate"] = pd.to_datetime(vintagedata["ReferenceDate"])

/var/folders/3k/vh6dl_9j30z3n567nqndm7tw0000gp/T/ipykernel_14664/1603484147.py:2: DtypeWarning: Columns (2,8,9,10,11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  vintagedata = pd.read_csv("/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/0_source/00_extract_vintagedata.csv")


In [17]:
to_write = {}

In [18]:
# Cards
contents = []

# ARIMA
reference_date = ar_sheets["Model Details"].loc[ar_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
forecast = ar_sheets["Forecast vs Actual"].loc[ar_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
lag = ar_sheets["Forecast vs Actual"].loc[ar_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

width = (ar_sheets['Confidence Bounds']["Predicted"] - ar_sheets['Confidence Bounds']["L1"]).tail(12).mean()

contents.append({
    "Card": ar_sheets["Model Details"].loc[ar_sheets["Model Details"]["Banner"] == "Model Name"]["Value"].item(),
    "Value": forecast,
    "Since Last Month": (forecast - lag) / lag,
    "Prediction Range": width,
})

# VAR
reference_date = var_sheets["Model Details"].loc[var_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
forecast = var_sheets["Forecast vs Actual"].loc[var_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
lag = var_sheets["Forecast vs Actual"].loc[var_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

width = (var_sheets['Confidence Bounds']["Predicted"] - var_sheets['Confidence Bounds']["L1"]).tail(12).mean()

contents.append({
    "Card": var_sheets["Model Details"].loc[var_sheets["Model Details"]["Banner"] == "Model Name"]["Value"].item(),
    "Value": forecast,
    "Since Last Month": (forecast - lag) / lag,
    "Prediction Range": width,
})

# ML
reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
forecast = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
lag = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

width = (ml_sheets['Confidence Bounds']["Predicted"] - ml_sheets['Confidence Bounds']["L1"]).tail(12).mean()

contents.append({
    "Card": "Confidence Interval",
    "Value": width,
    "Since Last Month": (forecast - lag) / lag,
    "Prediction Range": ""
})

to_write["Cards"] = pd.DataFrame(contents)

In [19]:
# Nowcast Browser – Header
reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
forecast = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
lag = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()
series_code = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Series Code"]["Value"].item()
LastUpdatedOnSource = pd.to_datetime(vintagedata["LastUpdatedOnSource"]).max()

contents = {
    "Value": forecast,
    "Since Last Month": (forecast - lag) / lag,
    "Series Name": variable.loc[variable["seriesid"] == series_code]["seriesname"].item(),
    "Series Code": series_code,
    "Reference Period": f"{reference_date.strftime('%b')} 1 - {reference_date.strftime('%b')} {pd.Period(reference_date.strftime('%Y-%m')).days_in_month}",
    "Region": variable.loc[variable["seriesid"] == series_code]["region"].item(),
    "Unit": variable.loc[variable["seriesid"] == series_code]["units"].item(),
    "Last Run Watermark": datetime.now().strftime('%d/%m/%Y %H:%M'),
    "Data as of": LastUpdatedOnSource.strftime('%d/%m/%Y %H:%M'),
    }

to_write["Nowcast Browser – Header"] = pd.DataFrame.from_dict(contents, orient="index").reset_index().rename(columns={"index": "Banner", 0: "Value"})

/var/folders/3k/vh6dl_9j30z3n567nqndm7tw0000gp/T/ipykernel_14664/3834755784.py:6: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  LastUpdatedOnSource = pd.to_datetime(vintagedata["LastUpdatedOnSource"]).max()


In [20]:
# Nowcast Browser – Base

reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()

traces = ml_sheets["Forecast vs Actual"].loc[
    (ml_sheets["Forecast vs Actual"]["Reference Date"] >= (reference_date - relativedelta(months=18))) &\
    (ml_sheets["Forecast vs Actual"]["Reference Date"] <= reference_date)
        ]

to_write["Nowcast Browser – Base"] = traces

In [21]:
traces.loc[(traces["Reference Date"] == reference_date), "Actual"] = None

In [22]:
traces

,Reference Date,Predicted,Actual
277,2023-02-01,15291.607834,15325.5
278,2023-03-01,15341.677061,15295.4
279,2023-04-01,15361.699079,15316.9
280,2023-05-01,15364.163441,15337.4
281,2023-06-01,15402.886122,15376.3
282,2023-07-01,15442.444731,15448.3
283,2023-08-01,15534.848933,15439.8
284,2023-09-01,15503.066251,15496.0
285,2023-10-01,15489.382602,15519.9
286,2023-11-01,15563.439473,15584.3


In [44]:
# Local Explanation

reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
lag_date = reference_date - relativedelta(months=1)
forecast = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date]["Predicted"].item()
lag = ml_sheets["Forecast vs Actual"].loc[ml_sheets["Forecast vs Actual"]["Reference Date"] == reference_date - relativedelta(months=1)]["Actual"].item()

impact_assessment = ml_sheets["Contributions"][["Unnamed: 0", "impact"]].rename(columns={"Unnamed: 0": "Series ID", "impact": "Impact"})

# values = ts.loc[ts["ReferenceDate"] == reference_date][list(impact_assessment["Series ID"])]
# values.index = ["Actual"]

# impact_assessment = impact_assessment.merge(values.T.reset_index().rename(columns={"index": "Series ID"}), how="left", on="Series ID")
impact_assessment["Impact"] = impact_assessment["Impact"].map(lambda x: "{:.2f}".format(x))
# impact_assessment["Actual"] = impact_assessment["Actual"].map(lambda x: "{:,.2f}".format(x))
# actuals = vintagedata.loc[
#     (vintagedata["VariableCode"].isin(list(impact_assessment["Series ID"]))) &\
#     (vintagedata["ReferenceDate"] <= reference_date)
#     ][["VariableCode", "Description", "ReferenceDate", "LastUpdatedOnSource"]].groupby(
#         ["VariableCode", "Description", "ReferenceDate"]
#         ).min().reset_index().rename(
#         columns={"VariableCode": "Series ID", "Description": "Data Series", "LastUpdatedOnSource": "Release Date"}
#         )

tmp = vintagedata.loc[
    (vintagedata["VariableCode"].isin(list(impact_assessment["Series ID"]))) &\
    (vintagedata["ReferenceDate"] <= reference_date)
    ]

actuals = tmp.merge(
    tmp.groupby(["VariableCode"]).agg({"ReferenceDate": "max"}).reset_index(), on=["VariableCode", "ReferenceDate"]
    )[["VariableCode", "Description", "ReferenceDate", "LastUpdatedOnSource"]].groupby(
        ["VariableCode", "Description", "ReferenceDate"]
        ).min().reset_index().rename(
        columns={"VariableCode": "Series ID", "Description": "Data Series", "LastUpdatedOnSource": "Release Date"}
        )

impact_assessment = impact_assessment.merge(
    actuals,
    on="Series ID",
    how="left"
)
impact_assessment["Release Date"] = pd.to_datetime(impact_assessment["Release Date"]).dt.strftime('%b-%d')

to_write["Local Explanation"] = impact_assessment[["Release Date", "Series ID", "Data Series", "Impact"]]

In [ ]:
# TODO: Global Explanation
series_code = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Series Code"]["Value"].item()
reference_date = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"] == "Reference Date"]["Value"].item()
df = ts.set_index("ReferenceDate")[list(impact_assessment["Series ID"])+[series_code]]
df.loc[reference_date, series_code] = None

# Melt wide DataFrame to long format
df_long = df.sort_index().loc[
    reference_date-relativedelta(months=6):reference_date
    ].reset_index().melt(
        id_vars=['ReferenceDate'], var_name='Variable Code', value_name='Variable Value'
        )

to_write["Global Explanation"] = df_long

In [52]:
df.loc[reference_date, series_code]

nan

In [55]:
# Model Assessment
df = ml_sheets["Model Details"].loc[ml_sheets["Model Details"]["Banner"].isin(["R-Squared", "MAPE"])].rename(columns={"Banner": "Measure"})
df["Measure"] = df["Measure"].replace({
    "R-Squared": "Adjusted R-Squared",
    "MAPE": "Average Error Rate"
})


to_write["Model Assessment"] = df

In [56]:
excel_file = "/Users/ejowik001/Desktop/Github/Nowcasting/kedro/refinery/data/08_reporting/dash_data_model.xlsx"

with pd.ExcelWriter(excel_file, engine="xlsxwriter") as writer:
    for sheet_name, contents in to_write.items():
        contents.to_excel(writer, sheet_name=sheet_name, index=False)